# 145 — LoRA Architecture Ablation

**What you'll learn:**
- The LoRA math: how rank `r` determines trainable parameter count
- What `target_modules` means and how to choose which layers to adapt
- How to run a sweep (ablation) across LoRA configs and compare results
- The tradeoff between rank size and training loss

---
**Source:** `examples/145-lora-architecture-ablation/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Part A** is CPU-safe. **Part B** requires a GPU for practical speed.

In [ ]:
%pip install -q transformers peft datasets torch

---
## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — The LoRA Math

Standard fine-tuning updates a weight matrix **W** of shape `(d_out, d_in)` — that's `d_out × d_in` parameters.

LoRA (Low-Rank Adaptation) *freezes* W and instead learns two small matrices:

```
ΔW = B × A

where:
  A ∈ R^(r × d_in)    ← r rows,  d_in columns
  B ∈ R^(d_out × r)   ← d_out rows, r columns

Trainable params per layer = r × d_in + d_out × r = r(d_in + d_out)

Full fine-tune params      = d_in × d_out
```

Because `r << d_in` and `r << d_out`, this is dramatically fewer parameters.

**Example:** For a query projection with `d_in = d_out = 512` and `r = 8`:
- Full: `512 × 512 = 262,144` params
- LoRA: `8 × 512 + 512 × 8 = 8,192` params ← **32× fewer**

The scaling factor `alpha` (usually `alpha = 2r`) controls how much the adapter output is weighted:
```
output = W·x + (alpha/r) × B·A·x
```
Setting `alpha = r` (scaling = 1.0) or `alpha = 2r` (scaling = 2.0) are common defaults.

In [ ]:
# A2 — Compute trainable params for different ranks (pure math, no model)

# Typical attention projection dimension for a small transformer
d = 512  # hidden size (d_in = d_out for square projections)

ranks = [4, 8, 16, 32]

# Configs mirroring workflow.py ABLATION_CONFIGS
ablation_configs = [
    {"r": 4,  "target_modules": ["q_proj"],            "label": "r4-q"},
    {"r": 8,  "target_modules": ["q_proj", "v_proj"],  "label": "r8-qv"},
    {"r": 16, "target_modules": ["q_proj", "v_proj"],  "label": "r16-qv"},
    {"r": 32, "target_modules": ["q_proj", "v_proj", "o_proj"], "label": "r32-qvo"},
]

# Assume the model has 12 transformer layers
n_layers = 12
full_proj_params = d * d  # params for one full projection matrix

print(f"Model: d={d}, {n_layers} layers")
print(f"Full projection W shape: ({d} x {d}) = {full_proj_params:,} params\n")

print(f"{'Config':<12} {'Rank':<6} {'Targets':<20} {'LoRA params':<14} {'vs Full fine-tune':<18} {'alpha'}")
print("-" * 76)

for cfg in ablation_configs:
    r = cfg["r"]
    n_mods = len(cfg["target_modules"])
    alpha = r * 2  # default: alpha = 2r

    # LoRA params per projection = r * d_in + d_out * r = 2*r*d (square case)
    lora_per_proj = 2 * r * d
    total_lora = lora_per_proj * n_mods * n_layers

    full_equivalent = full_proj_params * n_mods * n_layers
    reduction = full_equivalent / total_lora

    targets_str = "+".join(cfg["target_modules"])
    print(f"{cfg['label']:<12} {r:<6} {targets_str:<20} {total_lora:<14,} {reduction:<18.1f}x  {alpha}")

print()
print("LoRA params = r * (d_in + d_out) per adapted layer")
print("Higher rank = more capacity but more params and slower training")

### A3 — What `target_modules` Does

Each transformer layer has several weight matrices. `target_modules` selects which ones get a LoRA adapter.

```
Transformer self-attention block
──────────────────────────────────────────────────────────
  q_proj  →  projects input into Query space          ← common LoRA target
  k_proj  →  projects input into Key space
  v_proj  →  projects input into Value space          ← common LoRA target
  o_proj  →  projects attention output back           ← sometimes added

MLP block
  gate_proj  →  gated linear unit (gate)
  up_proj    →  gated linear unit (up)
  down_proj  →  projects back to model dimension
──────────────────────────────────────────────────────────
```

#### Typical strategies

| Strategy | Modules | Use case |
|----------|---------|----------|
| Minimal | `["q_proj", "v_proj"]` | Fastest, fewest params, works for many tasks |
| Extended | `["q_proj", "k_proj", "v_proj", "o_proj"]` | More expressive attention |
| Full attention + MLP | All 7 projections | Maximum capacity, used for domain adaptation |

**Rule of thumb:** Start with `q_proj + v_proj`. Add `k_proj` and `o_proj` if the task needs it.  
Adding MLP modules roughly 3× the trainable parameter count.

In [ ]:
# A4 — Print the 4 ablation configs from workflow.py as dict summaries

ABLATION_CONFIGS = [
    {"r": 4,  "target_modules": ["q_proj"],                     "label": "r4-q"},
    {"r": 8,  "target_modules": ["q_proj", "v_proj"],           "label": "r8-qv"},
    {"r": 16, "target_modules": ["q_proj", "v_proj"],           "label": "r16-qv"},
    {"r": 32, "target_modules": ["q_proj", "v_proj", "o_proj"], "label": "r32-qvo"},
]

print("LoRA ablation sweep — 4 configs\n")
for i, cfg in enumerate(ABLATION_CONFIGS):
    alpha = cfg["r"] * 2
    print(f"Config {i+1}: {cfg['label']}")
    print(f"  rank (r)        : {cfg['r']}")
    print(f"  lora_alpha      : {alpha} (= 2r, scaling factor = {alpha}/{cfg['r']} = 2.0)")
    print(f"  target_modules  : {cfg['target_modules']}")
    print(f"  lora_dropout    : 0.05")
    print(f"  bias            : 'none'")
    print()

print("What the sweep measures:")
print("  - Does higher rank always improve loss?")
print("  - Does adding o_proj help beyond q+v?")
print("  - What is the param-efficiency tradeoff?")

---
## Part B — Full Ablation Sweep

> **GPU RECOMMENDED**  
> This runs 4 sequential training jobs. On CPU each takes several minutes.  
> Free GPU: [Google Colab T4 runtime](https://colab.research.google.com) — Runtime > Change runtime type > T4 GPU.  
> Cells will be slow but not error on CPU for small models.

In [ ]:
# B1 — Full ablation sweep (from workflow.py)
import sys
sys.path.insert(0, '/content')  # Colab path; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "n_steps": 20,
    "results": [],
}

print(f"Running LoRA ablation on {state['model_name']}...")
print(f"4 configs × {state['n_steps']} steps each\n")
result = workflow(state)

print(f"\n{'Label':<12} {'Rank':<6} {'Targets':<30} {'Trainable%':<12} {'Loss'}")
print("-" * 72)
for r in result["results"]:
    targets = "+".join(r["targets"])
    pct = r["params"]["pct"]
    print(f"{r['label']:<12} {r['r']:<6} {targets:<30} {pct:<12.4f} {r['final_loss']}")

print("\nObservation: higher rank = more trainable params = typically lower loss,")
print("but with diminishing returns. Target module selection also matters.")